# Bài Tập Thực Hành: Phân Tích Dữ Liệu & Học Sâu (CNN)


## Import các thư viện cần thiết

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Cấu hình hiển thị
sns.set(style="whitegrid")
np.random.seed(42)  # để kết quả tái lập được


---
## PHẦN 1: KHỞI TẠO VÀ XỬ LÝ DỮ LIỆU



In [ ]:
n = 500

# Mã khách hàng: 'KH001' -> 'KH500'
ma_kh = ['KH' + str(i).zfill(3) for i in range(1, n + 1)]

# Tuổi: số nguyên ngẫu nhiên từ 18-70, chèn 10 giá trị NaN
tuoi = np.random.randint(18, 71, size=n).astype(float)
nan_idx_tuoi = np.random.choice(n, 10, replace=False)
tuoi[nan_idx_tuoi] = np.nan

# Thu nhập: số thực từ 5 triệu - 50 triệu, thêm 5 outlier lên tới 200 triệu
thu_nhap = np.random.uniform(5_000_000, 50_000_000, size=n)
outlier_idx = np.random.choice(n, 5, replace=False)
thu_nhap[outlier_idx] = np.random.uniform(150_000_000, 200_000_000, size=5)

# Giới tính: ngẫu nhiên Nam/Nữ, chèn 15 giá trị NaN
gioi_tinh = np.random.choice(['Nam', 'Nữ'], size=n).astype(object)
nan_idx_gt = np.random.choice(n, 15, replace=False)
gioi_tinh[nan_idx_gt] = np.nan

# Thành phố: ngẫu nhiên trong 3 thành phố
thanh_pho = np.random.choice(['Hà Nội', 'Đà Nẵng', 'TP.HCM'], size=n)

# Tổng chi tiêu: tương quan nhẹ với thu nhập (30% thu nhập + nhiễu ngẫu nhiên)
tong_chi_tieu = thu_nhap * 0.3 + np.random.normal(0, 3_000_000, size=n)
tong_chi_tieu = np.clip(tong_chi_tieu, 500_000, None)  # không âm

df_khachhang = pd.DataFrame({
    'MaKH': ma_kh,
    'Tuoi': tuoi,
    'ThuNhap': thu_nhap,
    'GioiTinh': gioi_tinh,
    'ThanhPho': thanh_pho,
    'TongChiTieu': tong_chi_tieu
})

print(df_khachhang.shape)
df_khachhang.head()


### Câu 2: Xử lý giá trị khuyết (Missing Values)



In [ ]:
# Kiểm tra số lượng giá trị khuyết trên mỗi cột
print("Số giá trị khuyết theo từng cột:")
print(df_khachhang.isnull().sum())

# Điền khuyết cột Tuoi bằng Trung vị
median_tuoi = df_khachhang['Tuoi'].median()
df_khachhang['Tuoi'] = df_khachhang['Tuoi'].fillna(median_tuoi)

# Điền khuyết cột GioiTinh bằng Mode (giá trị xuất hiện nhiều nhất)
mode_gioitinh = df_khachhang['GioiTinh'].mode()[0]
df_khachhang['GioiTinh'] = df_khachhang['GioiTinh'].fillna(mode_gioitinh)

print("\nSau khi xử lý:")
print(df_khachhang.isnull().sum())


### Câu 3: Mã hóa biến phân loại (Categorical Encoding)




In [ ]:
df_thanhpho_encoded = pd.get_dummies(df_khachhang['ThanhPho'], prefix='ThanhPho')
df_khachhang = pd.concat([df_khachhang, df_thanhpho_encoded], axis=1)

df_khachhang.head()


### Câu 4: Phát hiện và xử lý điểm dị biệt (Outlier Detection - IQR)




In [ ]:
Q1 = df_khachhang['ThuNhap'].quantile(0.25)
Q3 = df_khachhang['ThuNhap'].quantile(0.75)
IQR = Q3 - Q1

gioi_han_duoi = Q1 - 1.5 * IQR
gioi_han_tren = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:,.0f}, Q3 = {Q3:,.0f}, IQR = {IQR:,.0f}")
print(f"Giới hạn dưới = {gioi_han_duoi:,.0f}, Giới hạn trên = {gioi_han_tren:,.0f}")

so_dong_truoc = df_khachhang.shape[0]

df_khachhang = df_khachhang[
    (df_khachhang['ThuNhap'] >= gioi_han_duoi) &
    (df_khachhang['ThuNhap'] <= gioi_han_tren)
].reset_index(drop=True)

so_dong_sau = df_khachhang.shape[0]
print(f"Số dòng trước khi lọc: {so_dong_truoc}, sau khi lọc: {so_dong_sau}")


### Câu 5: Chuẩn hóa dữ liệu (Feature Scaling)



In [ ]:
scaler = MinMaxScaler()
df_khachhang['TongChiTieu_Scaled'] = scaler.fit_transform(df_khachhang[['TongChiTieu']])

df_khachhang[['TongChiTieu', 'TongChiTieu_Scaled']].head()


### Câu 6: Lọc dữ liệu theo điều kiện (Data Filtering)



In [ ]:
df_loc = df_khachhang[
    (df_khachhang['GioiTinh'] == 'Nữ') &
    (df_khachhang['Tuoi'] > 30) &
    (df_khachhang['ThanhPho'] == 'Hà Nội')
]

print(f"Số khách hàng thỏa điều kiện: {df_loc.shape[0]}")
df_loc.head()


### Câu 7: Gom nhóm và Thống kê (Aggregation)



In [ ]:
thong_ke_thanhpho = df_khachhang.groupby('ThanhPho')['TongChiTieu'].agg(['mean', 'sum'])
thong_ke_thanhpho.columns = ['TrungBinh_ChiTieu', 'Tong_ChiTieu']
thong_ke_thanhpho


### Câu 8: Kỹ nghệ đặc trưng (Feature Engineering)



In [ ]:
bins = [17, 30, 45, 60, 200]
labels = ['18-30', '31-45', '46-60', 'Trên 60']

df_khachhang['NhomTuoi'] = pd.cut(df_khachhang['Tuoi'], bins=bins, labels=labels)

df_khachhang[['Tuoi', 'NhomTuoi']].head(10)


### Câu 9: Ma trận tương quan (Correlation Matrix)



In [ ]:
corr_matrix = df_khachhang[['Tuoi', 'ThuNhap', 'TongChiTieu']].corr(method='pearson')
print(corr_matrix)

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Ma trận tương quan Pearson')
plt.tight_layout()
plt.show()


### Câu 10: Trực quan hóa dữ liệu (Scatter Plot)

Mối quan hệ giữa `ThuNhap` (X) và `TongChiTieu` (Y), tô màu theo `GioiTinh`.


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_khachhang, x='ThuNhap', y='TongChiTieu', hue='GioiTinh', alpha=0.7)
plt.title('Mối quan hệ giữa Thu nhập và Tổng chi tiêu')
plt.xlabel('Thu nhập (VNĐ)')
plt.ylabel('Tổng chi tiêu (VNĐ)')
plt.tight_layout()
plt.show()


---
## PHẦN 2: ỨNG DỤNG HỌC SÂU - MẠNG NƠ-RON TÍCH CHẬP (CNN)




In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)


#### 1. Tiền xử lý dữ liệu

In [ ]:
# Tải bộ dữ liệu Fashion MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

print("Kích thước tập train:", X_train.shape)
print("Kích thước tập test:", X_test.shape)

# Chuẩn hóa (scale) giá trị pixel về khoảng [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape dữ liệu để thêm kênh màu (28, 28, 1) - vì ảnh xám chỉ có 1 kênh
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print("Sau khi reshape:", X_train.shape, X_test.shape)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Xem thử vài ảnh mẫu
plt.figure(figsize=(8, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()


#### 2. Xây dựng kiến trúc CNN

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()


#### 3. Biên dịch và Huấn luyện mô hình

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=5,
    validation_split=0.1,
    batch_size=64
)


#### 4. Đánh giá mô hình trên tập kiểm thử

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"\nĐộ chính xác (Accuracy) trên tập test: {test_accuracy:.4f}")
print(f"Loss trên tập test: {test_loss:.4f}")


In [ ]:
# (Tùy chọn) Trực quan hóa quá trình huấn luyện
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Quá trình huấn luyện CNN trên Fashion MNIST')
plt.legend()
plt.tight_layout()
plt.show()
